# 07. Reusable Builder Preview Validation

`00~06`에서 결정한 내용을 실제 재사용 스크립트에 태워보는 노트북임.

여기서는 full dataset을 바로 만들지 않고, scene별 소량 preview dataset만 생성함. 목적은 세 가지임.

1. `30_pipelines/culane_pseudo_dataset_builder`의 재사용 빌더가 현재 결정값을 제대로 읽는지 확인함.
2. 생성된 CULane-style 구조가 `train_gt.txt`, `val.txt`, `.lines.txt`, mask까지 깨지지 않는지 검증함.
3. full build 전에 output 구조와 summary를 눈으로 확인함.

중요한 의사결정은 이미 `06`까지 끝났음. 이 노트북부터는 “결정”보다 “재현 가능한 도구 검증”에 가깝다.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path('~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization')
PIPELINE_ROOT = PROJECT_ROOT / "30_pipelines" / "culane_pseudo_dataset_builder"
CONFIG_PATH = PIPELINE_ROOT / "configs" / "project_map_field2_component_poly.json"
PREVIEW_CONFIG_PATH = PIPELINE_ROOT / "outputs" / "project_map_field2_component_poly_preview_max3.json"
PREVIEW_DATASET_ROOT = PIPELINE_ROOT / "outputs" / "preview_map_culane_field2_component_poly_max3"

BUILD_SCRIPT = PIPELINE_ROOT / "src" / "build_culane_pseudo_dataset.py"
VALIDATE_SCRIPT = PIPELINE_ROOT / "src" / "validate_culane_dataset.py"

assert CONFIG_PATH.exists(), CONFIG_PATH
assert BUILD_SCRIPT.exists(), BUILD_SCRIPT
assert VALIDATE_SCRIPT.exists(), VALIDATE_SCRIPT

print(f"python: {sys.executable}")
print(f"config: {CONFIG_PATH}")
print(f"preview output: {PREVIEW_DATASET_ROOT}")


python: ~\anaconda3\envs\<env>\python.exe
config: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\configs\project_map_field2_component_poly.json
preview output: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\outputs\preview_map_culane_field2_component_poly_max3


## 1. 확정 설정 확인

현재 config는 `03~05`에서 확정한 HSV와 lane extraction method를 담고 있음.

- HSV: 노란 차선을 mask로 잡기 위한 색상 범위
- `component_poly`: mask에서 연결 성분을 분리하고, 각 물리 차선 후보를 polynomial 형태의 `.lines.txt` 좌표로 변환하는 방식
- quality gate: 자동 학습 후보와 review 후보를 나누는 최소 품질 기준


In [2]:
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

summary = {
    "name": config["name"],
    "manifest_csv": config["input"]["manifest_csv"],
    "output_dataset_root": config["output"]["dataset_root"],
    "camera": config["camera"],
    "hsv": config["hsv"],
    "lane_extraction": config["lane_extraction"],
    "quality_gate": config["quality_gate"],
    "review_priority_policy": config["review_priority_policy"],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "name": "project_map_field2_component_poly",
  "manifest_csv": "10_experiments/04_field2_map_lab/field2_experiment/data/20260501_field2/manifests/01_lane_drive_capture_manifest.csv",
  "output_dataset_root": "30_pipelines/culane_pseudo_dataset_builder/datasets/map_culane_field2_component_poly",
  "camera": {
    "raw_width": 1296,
    "raw_height": 972,
    "cut_height": 445,
    "model_width": 800,
    "model_height": 320,
    "num_points": 72,
    "max_lanes": 4
  },
  "hsv": {
    "lower": [
      22,
      90,
      110
    ],
    "upper": [
      38,
      255,
      255
    ],
    "morph_kernel": 5
  },
  "lane_extraction": {
    "method": "component_poly",
    "y_step": 6,
    "min_run_width": 4,
    "min_points": 8,
    "min_area": 450,
    "min_height": 120,
    "degree": 2
  },
  "quality_gate": {
    "min_lanes": 1,
    "min_point_total": 12,
    "min_y_span": 80,
    "min_mask_support": 0.2,
    "reject_crossing": false,
    "reject_small_gap": false
  },
  "review_prio

## 2. Preview 전용 config 생성

원본 config는 full build용으로 유지함. 이 노트북은 실수로 full output을 덮어쓰지 않도록 preview output으로만 빌드함.


In [3]:
preview_config = json.loads(json.dumps(config, ensure_ascii=False))
preview_config["name"] = config["name"] + "_preview_max3"
preview_config["output"]["dataset_root"] = "30_pipelines/culane_pseudo_dataset_builder/outputs/preview_map_culane_field2_component_poly_max3"
preview_config["reports_dir"] = "30_pipelines/culane_pseudo_dataset_builder/reports"

PREVIEW_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(PREVIEW_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(preview_config, f, ensure_ascii=False, indent=2)

print(PREVIEW_CONFIG_PATH)


~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\outputs\project_map_field2_component_poly_preview_max3.json


## 3. Preview dataset 생성

`--max-per-scene 3`은 각 scene에서 3장씩만 뽑는다는 뜻임. 즉, 빠르게 구조만 확인하는 preview임.


In [4]:
def run_json_command(cmd):
    print(" ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"command failed: {result.returncode}")
    return json.loads(result.stdout)

build_cmd = [
    sys.executable,
    str(BUILD_SCRIPT),
    "--config", str(PREVIEW_CONFIG_PATH),
    "--max-per-scene", "3",
    "--clean",
]
build_summary = run_json_command(build_cmd)


~\anaconda3\envs\<env>\python.exe ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\src\build_culane_pseudo_dataset.py --config ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\outputs\project_map_field2_component_poly_preview_max3.json --max-per-scene 3 --clean


{
  "name": "project_map_field2_component_poly_preview_max3",
  "output_root": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\30_pipelines\\culane_pseudo_dataset_builder\\outputs\\preview_map_culane_field2_component_poly_max3",
  "input_rows": 27,
  "status_counts": {
    "usable": 27
  },
  "scene_status_counts": {
    "failure_cases::usable": 3,
    "intersection_approach::usable": 3,
    "intersection_left::usable": 3,
    "intersection_right::usable": 3,
    "left_curve::usable": 3,
    "off_center_left::usable": 3,
    "off_center_right::usable": 3,
    "right_curve::usable": 3,
    "straight::usable": 3
  },
  "review_priority_counts": {
    "hard_case": 3,
    "high": 9,
    "normal": 15
  },
  "train_rows": 21,
  "val_rows": 6,
  "manifest": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\30_pipelines\\culane_pseudo_dataset_builder\\outputs\\preview_map_culane_field2_component_poly_max3\\build_manifest.csv",
  "geometry": {
    "

## 4. Preview dataset 검증

검증은 CULane-style 파일들이 실제로 맞물리는지 확인함.

- list 파일 존재 여부
- 이미지 존재 여부
- `.lines.txt` 존재 및 비어있지 않은지
- segmentation mask 존재 여부
- 이미지/mask 해상도가 1296x972인지


In [5]:
validate_cmd = [
    sys.executable,
    str(VALIDATE_SCRIPT),
    "--dataset-root", str(PREVIEW_DATASET_ROOT),
    "--config", str(PREVIEW_CONFIG_PATH),
]
validation_summary = run_json_command(validate_cmd)


~\anaconda3\envs\<env>\python.exe ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\src\validate_culane_dataset.py --dataset-root ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\outputs\preview_map_culane_field2_component_poly_max3 --config ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\30_pipelines\culane_pseudo_dataset_builder\outputs\project_map_field2_component_poly_preview_max3.json


{
  "dataset_root": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\30_pipelines\\culane_pseudo_dataset_builder\\outputs\\preview_map_culane_field2_component_poly_max3",
  "geometry": {
    "raw_width": 1296,
    "raw_height": 972,
    "cut_height": 445,
    "model_width": 800,
    "model_height": 320,
    "num_points": 72,
    "max_lanes": 4,
    "visible_height": 527
  },
  "counts": {
    "train_gt.txt": 21,
    "val.txt": 6,
    "test.txt": 6
  },
  "issue_count": 0,
  "issues_sample": []
}



## 5. 결과 요약 확인


In [6]:
print("build summary")
print(json.dumps({
    "input_rows": build_summary.get("input_rows"),
    "status_counts": build_summary.get("status_counts"),
        "review_priority_counts": build_summary.get("review_priority_counts"),
    "train_rows": build_summary.get("train_rows"),
    "val_rows": build_summary.get("val_rows"),
    "lane_extraction": build_summary.get("lane_extraction"),
        "quality_gate": build_summary.get("quality_gate"),
}, ensure_ascii=False, indent=2))

print()
print("validation summary")
print(json.dumps({
    "counts": validation_summary.get("counts"),
    "issue_count": validation_summary.get("issue_count"),
    "issues_sample": validation_summary.get("issues_sample"),
}, ensure_ascii=False, indent=2))


build summary
{
  "input_rows": 27,
  "status_counts": {
    "usable": 27
  },
  "review_priority_counts": {
    "hard_case": 3,
    "high": 9,
    "normal": 15
  },
  "train_rows": 21,
  "val_rows": 6,
  "lane_extraction": {
    "method": "component_poly",
    "y_step": 6,
    "min_run_width": 4,
    "min_points": 8,
    "min_area": 450,
    "min_height": 120,
    "degree": 2
  },
  "quality_gate": {
    "min_lanes": 1,
    "min_point_total": 12,
    "min_y_span": 80,
    "min_mask_support": 0.2,
    "reject_crossing": false,
    "reject_small_gap": false
  }
}

validation summary
{
  "counts": {
    "train_gt.txt": 21,
    "val.txt": 6,
    "test.txt": 6
  },
  "issue_count": 0,
  "issues_sample": []
}


## 07 결과 메모

Preview build/validation까지 정상 통과함.

- preview 입력: `27장`
- status: `{'usable': 27}`
- review priority: `{'hard_case': 3, 'high': 9, 'normal': 15}`
- train rows: `21`
- val/test rows: `6`
- validation issue: `0`
- method: `component_poly`
- quality gate: `{'min_lanes': 1, 'min_point_total': 12, 'min_y_span': 80, 'min_mask_support': 0.2, 'reject_crossing': False, 'reject_small_gap': False}`

이 결과는 `06`에서 확정한 자동 reject gate가 재사용 빌더에서도 깨지지 않는다는 뜻임. Preview에서는 각 scene 3장씩 총 27장이 모두 usable로 남았고, 교차로/high priority도 학습 후보에 포함되는 구조가 확인됨.

단, 여기서 만든 dataset은 preview임. 학습에 바로 넣을 최종 dataset은 `08`에서 full build로 생성해야 함.
